# Pairing and superfluidity in the attractive Hubbard model

This notebook uses [pyALF](https://github.com/ALF-QMC/pyALF) to study the **attractive Hubbard model on the square lattice**. We cool systems of several sizes and measure the on-site pair correlation function. This illustrates the development of correlations between electron pairs and introduces the physics behind the BEC terminology used for strongly bound fermion pairs.

## The model and its transition

The Hamiltonian is

$$
\hat H=-t\sum_{\langle i,j\rangle,\sigma}
\left(\hat c^\dagger_{i\sigma}\hat c_{j\sigma}+\mathrm{h.c.}\right)
+U\sum_i\left(\hat n_{i\uparrow}-\frac12\right)
\left(\hat n_{i\downarrow}-\frac12\right)
-\mu\sum_{i,\sigma}\hat n_{i\sigma},\qquad U<0.
$$

Here $t$ is the nearest-neighbour hopping, $U$ the on-site attraction, and $\mu$ the chemical potential. We use $t=1$ and $k_{\mathrm B}=1$. The simulation fixes $U/t=-4$ and $\mu/t=0.5$, and varies $T=1/\beta$. In this particle-hole-symmetric convention, half filling occurs at $\mu=0$; the chosen positive chemical potential favours densities above one electron per site. The density is measured rather than fixed.

Attraction favours on-site spin-singlet pairs. As the attraction grows, overlapping Cooper pairs evolve smoothly into tightly bound pairs that behave as composite bosons: the **BCS–BEC crossover**. Pair formation and collective phase coherence are distinct phenomena; pairs can exist without coherent superfluid behaviour. This notebook scans temperature at one interaction strength, so it does not map out that crossover. See [Dupuis (2004)](https://arxiv.org/abs/cond-mat/0311374).

In this two-dimensional model, the finite-temperature superfluid transition away from half filling is a **Berezinskii–Kosterlitz–Thouless (BKT) transition**. Below it, vortex–antivortex pairs are bound and pair correlations decay algebraically; above it, free vortices destroy this algebraic order. Thus the finite-temperature phase has quasi-long-range order, rather than a nonzero condensate density in the thermodynamic limit. At half filling, an enlarged symmetry relates pairing and charge-density-wave order and suppresses the transition temperature to zero. See [Paiva et al. (2004)](https://arxiv.org/abs/cond-mat/0403397).

The short runs and small lattices below demonstrate how to measure pairing correlations. Their curves alone cannot establish a transition temperature.


## Import the tools

`Simulation` prepares and runs ALF, `ALF_source` manages its source checkout, and `load_res` reads the analysed results. `Lattice` maps physical momenta to array indices; NumPy and pandas organise the data, and Matplotlib makes the plots.

The plotting setup uses `scienceplots` for styling and `%matplotlib widget` for interactive figures, which requires `ipympl` in the notebook environment. The `science` style also enables LaTeX rendering by default. If those optional plotting dependencies are unavailable, a standard Matplotlib style and inline backend can be used instead. The ALF build requirements are described in the [installation section](../02-installation.md).


In [1]:
import matplotlib.pyplot as plt  # Plotting library
%matplotlib widget
import scienceplots
plt.style.use('science')

import numpy as np  # Numerical libary
from py_alf import ALF_source, Simulation, Lattice
from py_alf.ana import load_res  # Function for loading analysis results
from py_alf.utils import find_sim_dirs  # Function for finding QMC bins

from pathlib import Path
import pandas as pd
from collections import defaultdict

## Set the ALF source directory

The notebook expects the `ALF` directory in the notebook's working directory. If it is not present it will automatically download it. `ALF_source` manages this checkout, and the returned `alf_dir` object is passed to every simulation.

**Required observable:** this tutorial depends on an ALF extension that measures the native `Pair` correlator and writes `Pair_eq`. The Python analysis stores its momentum-space results under `Pair_eqK` and `Pair_eqK_err`.


In [2]:
alf_dir = ALF_source(alf_dir=str(Path.cwd() / 'ALF'), branch="s-wave-Pairing_correlations")

Checking out branch s-wave-Pairing_correlations
Your branch is up to date with 'origin/s-wave-Pairing_correlations'.


Already on 's-wave-Pairing_correlations'


## Compile ALF

Create a Hubbard `Simulation` object and compile the executable once. Compilation prepares the program; it does not yet run the Monte Carlo calculation. The same executable is reused for every temperature and lattice size.

In [6]:
compile_sim = Simulation(alf_src=alf_dir, ham_name='Hubbard',sim_dict={}) #sim_dict is used to choose parameters.
compile_sim.compile()

Checking out branch s-wave-Pairing_correlations
Your branch is up to date with 'origin/s-wave-Pairing_correlations'.


Already on 's-wave-Pairing_correlations'
========== Downloading HDF5 source ==========
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 37.4M  100 37.4M    0     0  25.5M      0  0:00:01  0:00:01 --:--:-- 25.5M
=== Build with the following compilers C: gcc, Fortran: gfortran, C++: g++ 
configure: WARNING:
           Fortran REAL(KIND=16) is 16 Bytes, but no corresponding C float type exists of that size
                     !!! Fortran interfaces will not be generated for REAL(KIND=16) !!!
          
sed: 1: "fortran/src/H5config_f. ...": invalid command code f
In file included from ../hdf5-1.14.6/src/H5B.c:100:
../hdf5-1.14.6/src/H5B.c: In function 'H5B__remove_helper':
../hdf5-1.14.6/src/H5B.c:1427:46: warning: potential null pointer dereference [-Wnull-dereference]
 1427 |     if (*lt_key_changed && H5_addr_defined(bt->left)) {
../hdf5-1.14.6/src/H5private.h:449:36: note:

Compiling ALF... 
Cleaning up Prog/
Cleaning up Libraries/
Cleaning up Analysis/
Compiling Libraries


entanglement_mod.F90:35:2:

   35 | #warning "You are compiling entanglement without MPI. No Renyi entropy results possible, all other observables still work!"
      |  1~~~~~~
ar: creating archive modules_90.a
ar: creating archive libqrref.a


Compiling Analysis
Compiling Program
Compiling program modules
Parsing Hamiltonian parameters
filenames: Hamiltonians/Hamiltonian_Kondo_smod.F90 Hamiltonians/Hamiltonian_Kondo_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_smod.F90 Hamiltonians/Hamiltonian_Hubbard_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_smod.F90 Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_tV_smod.F90 Hamiltonians/Hamiltonian_tV_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_LRC_smod.F90 Hamiltonians/Hamiltonian_LRC_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Z2_Matter_smod.F90 Hamiltonians/Hamiltonian_Z2_Matter_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Spin_Peierls_smod.F90 Hamiltonians/Hamiltonian_Spin_Peierls_read_write_parameters.F90
Link program
Done.


## Run the simulations

We simulate periodic square lattices with $L=4,6,8$, containing $N_s=L^2=16,36,64$ sites, where the inverse temperatures takes values $\beta t=1,2,\ldots,7$, thus yielding 21 independent Simulations in total.

| Parameter | Meaning in this example |
| --- | --- |
| `Lattice_type`, `L1`, `L2` | Square geometry with $L\times L$ sites and one site per unit cell. |
| `ham_U=-4.0` | On-site attraction of magnitude $4t$. |
| `ham_chem=0.5` | Fixed chemical potential; the particle density may vary with temperature and size. |
| `Beta` | Inverse temperature $\beta=1/T$. Larger values mean lower temperatures. |
| `Nsweep`, `NBin` | Number of Monte Carlo sweeps per bin and Number of bins per simulation. |
| `Mz=True` | A Hubbard–Stratonovich decomposition in the spin-density channel. This selects the auxiliary-field representation, not an imposed magnetic order. |
| `N_SUN=2`, `N_FL=2` | The physical spin-$\tfrac12$ model. In this two-flavour Hubbard setup, ALF internally halves `N_SUN`, leaving one colour for each spin flavour. |
| `Checkerboard`, `Symm` | A symmetric checkerboard decomposition of imaginary-time evolution; finite-time-step errors remain to be checked. |
| `Ltau=0` | Measure equal-time observables only, which suffice for the pair structure factor. |

The dictionary leaves the hopping and time step at their current defaults, $t=1$ and $\Delta\tau=0.1$. For each $(L,\beta)$, `sim.run()` performs the simulation and `sim.analysis()` estimates means and statistical errors. Results are stored separately by size and inverse temperature. Existing run directories can be resumed, so the saved statistics may include earlier runs as well as the requested new bins.


In [ ]:
LAT_SIZES = [4,6,8]
BETAS = [i for i in range(1,8)]

for beta in BETAS:
    for L in LAT_SIZES:
        sim = Simulation(
                    alf_dir, 'Hubbard',
                    {
                        ### Choice of Predefined Model###
                        'Model': 'Hubbard',
                        
                        ###Definition of the Lattice###
                        'Lattice_type': 'Square',
                        'L1': L,
                        'L2': L,
                        
                        #--- Model parameters ---#
                        'Beta': float(beta),
                        'ham_chem': 0.5,
                        'ham_U': -4.0,
                        #--- Simulation parameters --- #
                        'Nsweep': 10,
                        'NBin': 10,
                        'Mz': True, #Choice of HS-Transformation (Exclusive for Hubbard-type interactions); ALF-DOC: 8.3.2 M_Z-Hubbard interaction
                        'Checkerboard': True, #ALF-DOC: 2.3. The Trotter error and checkerboard decomposition
                        'Symm': True, #Symmetric Trotter decomposition; ALF-DOC: 2.3. The Trotter error and checkerboard decomposition

                        ###ALF Internal Flags###
                        'N_SUN': 2, #
                        'N_FL': 2, # 
                        'Ltau': 0, #Speed up simulation by not calculating time-displaced observables, which take more time to compute.
                    },
                    machine='GNU',
                    sim_dir=f'Attractive_Hubbard_L{L}_beta_{beta}'
                )
        sim.run()
        sim.analysis()
print('All simulations and analyses complete.')

Prepare directory "/Users/luis/Documents/_WORK/CONFERENCES/ALF_WORKSHOP/ALF_Tutorial/jupyterbook/source/04-nice-physics/ALF_data/Attractive_Hubbard_L4_beta_1" for Monte Carlo run.
Resuming previous run.
Run /Users/luis/Documents/_WORK/CONFERENCES/ALF_WORKSHOP/ALF_Tutorial/jupyterbook/source/04-nice-physics/ALF/Prog/ALF.out
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; for details see license.GPL
 This is free software, and you are welcome to redistribute it under certain conditions.
### Analyzing /Users/luis/Documents/_WORK/CONFERENCES/ALF_WORKSHOP/ALF_Tutorial/jupyterbook/source/04-nice-physics/ALF_data/Attractive_Hubbard_L4_beta_1 ###
/Users/luis/Documents/_WORK/CONFERENCES/ALF_WORKSHOP/ALF_Tutorial/jupyterbook/source/04-nice-physics
Scalar observables:
Ener_scal
Kin_scal
Part_scal
Pot_scal
Histogram observables:
Equal time observables:
Den_eq
Green_eq
Pair_eq
SpinT_eq
SpinXY_eq
SpinZ_eq
Time displaced observables:
Prepar

## Load the analysed observables

`find_sim_dirs()` discovers simulation directories and `load_res(dirs)` collects their analysed results into a pandas table, with one row per run. Restrict `dirs` to this temperature scan if the working directory also contains runs from the Mott notebook or other examples: those runs need not contain a `Pair` observable.

The analysis below reads:

- `Pair_eqK` and `Pair_eqK_err`: the pair correlation matrix in momentum space and its statistical error.
- `Pair_eq_lattice`: the geometry needed to locate a momentum in the stored arrays.
- `Part_scal0` and `Part_scal0_err`: the total particle number and its error in the standard Hubbard measurement.
- `l1` and `beta`: the lattice size and inverse temperature identifying each result.

The suffix `eq` denotes equal imaginary time, while `K` denotes momentum space. An equal-time structure factor is different from a pair susceptibility, which also involves an imaginary-time integral.


In [ ]:
dirs = find_sim_dirs()
res=load_res(dirs)

./ALF_data/Attractive_Hubbard_L4_beta_1
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_2
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_3
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_4
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_5
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_6
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L4_beta_7
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_1
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_2
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_3
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_4
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_5
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_6
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L6_beta_7
No orbital locations saved.
./ALF_data/Attractive_Hubbard_L8_beta_1
No orbit

## Extract the pair structure factor

Define the on-site spin-singlet pair operators

$$
\hat b_i^\dagger=\hat c_{i\uparrow}^\dagger\hat c_{i\downarrow}^\dagger,
\qquad
\hat b_i=\hat c_{i\downarrow}\hat c_{i\uparrow}.
$$

The correlator $\langle\hat b_i^\dagger\hat b_j\rangle$ measures coherence between pairs at different sites. The on-site orbital structure is invariant under lattice rotations, giving this pairing channel its $s$-wave character. We use the structure-factor convention

$$
N(\mathbf q)=\frac{1}{N_s}\sum_{i,j}
e^{i\mathbf q\cdot(\mathbf r_i-\mathbf r_j)}
\langle\hat b_i^\dagger\hat b_j\rangle.
$$

Uniform pairing is detected at $\mathbf q=(0,0)$, where contributions from distant sites add without a relative phase. Even incoherent local pairs contribute to $N(0)$: the terms with $i=j$ are the double occupancy $\langle\hat n_{i\uparrow}\hat n_{i\downarrow}\rangle$. A nonzero value alone therefore does not establish superfluidity.

`extract_pair_at_q` uses `Lattice.k_to_n` to locate the requested momentum. The square lattice has one site per unit cell, so `np.squeeze` removes the singleton orbital axes and leaves one value and error per run. The function groups these data, together with the particle number and inverse temperature, by lattice size.

<!-- The helper plots the stored `Pair_eqK` value without any additional normalization. Its interpretation as $N(\mathbf q)$ assumes that the custom `Pair` implementation follows the definition above. Check that measurement's normalization and operator ordering before comparing absolute values with another code or a symmetrized correlator such as $\langle\hat b_i^\dagger\hat b_j+\hat b_j^\dagger\hat b_i\rangle$. -->


In [ ]:
def extract_pair_at_q(q: tuple, res: pd.DataFrame):
    plot_data = defaultdict(lambda: defaultdict(list))
    for idx in range(len(res)):
        latt = Lattice(res.iloc[idx]["Pair_eq_lattice"]) 
        q_idx = latt.k_to_n(q)
        pair_eq_k = np.squeeze(res.iloc[idx]["Pair_eqK"][...,q_idx])# squeeze because returned shape was (1,1,num_k) 
        pair_eq_k_err = np.squeeze(res.iloc[idx]["Pair_eqK_err"][...,q_idx]) # squeeze because returned shape was (1,1,num_k)
        L = res.iloc[idx]["l1"]
        beta = res.iloc[idx]["beta"]
        part_num_val = res.iloc[idx]["Part_scal0"]
        part_num_err = res.iloc[idx]["Part_scal0_err"]

        plot_data[f"L_{L}"]["beta"].append(beta)
        plot_data[f"L_{L}"]["pair_value"].append(pair_eq_k)
        plot_data[f"L_{L}"]["pair_err"].append(pair_eq_k_err)        
        plot_data[f"L_{L}"]["part_num_value"].append(part_num_val)
        plot_data[f"L_{L}"]["part_num_err"].append(part_num_err)
    return plot_data

### Select zero momentum

Evaluate the helper at $\mathbf q=(0,0)$ to obtain the uniform pairing signal for each $(L,\beta)$.

In [ ]:
plot_data = extract_pair_at_q((0,0), res)

## Plot the pairing correlations

Convert each inverse temperature to $T=1/\beta$ and plot $N(0)$ with statistical error bars. Since $t=1$, the horizontal axis is numerically $T/t$. Cooling moves from right to left on this plot. Growth upon cooling and an increasing dependence on $L$ are signs that pair correlations extend over larger distances; their interpretation requires a comparison of several system sizes.


In [ ]:
plt.figure(figsize=(8,6))
for L_key in plot_data.keys():
    T = [1/beta for beta in plot_data[L_key]["beta"]]
    data = plot_data[L_key]["pair_value"]
    err = plot_data[L_key]["pair_err"]
    plt.errorbar(T, data, yerr=err, label=L_key)
plt.title(r"$s$-wave pairing correlation at $\mu = 0.5$")
plt.xlabel(r"T")
plt.ylabel(r"$N(q=(0,0))$")
plt.legend()
plt.show()

## Monitor the particle number

At fixed chemical potential, the particle number can change as the system is cooled. The second plot tracks this change and helps determine the filling at which the pairing correlations were measured.

The standard ALF Hubbard observable `Part_scal0` is the **total** particle number,

$$
N_e=\sum_{i,\sigma}\langle\hat n_{i\sigma}\rangle.
$$

This is what the existing plotting cell shows. To compare fillings across lattice sizes, use the density per site $n=N_e/L^2$, with error $\delta n=\delta N_e/L^2$. Then $0\leq n\leq2$, and half filling is $n=1$. Raw particle numbers increase with the number of sites even when the density is unchanged.



In [ ]:
plt.figure(figsize=(8,6))
for L_key in plot_data.keys():
    data = plot_data[L_key]["part_num_value"]
    err = plot_data[L_key]["part_num_err"]
    plt.errorbar(T, data, yerr=err, label=L_key)
    plt.title(r"Occupation number $\mu = 0.5$")
    plt.xlabel(r"T")
    plt.ylabel(r"$\langle n \rangle = \sum_{i,\sigma} \langle c^\dagger_{i,\sigma} c_{i,\sigma} \rangle$")
plt.legend()
plt.show()